# lists
Lists are Python’s default mutable sequence for ordered collections. In engineering systems they show up as buffers, batches, queues-in-practice, parsed rows, transformed records, and temporary working sets. The main skill is understanding when mutability helps and when it leaks bugs.

## 1. Problem First
Why does this matter in real systems?
- API results often arrive as lists of records.
- Batch jobs build and flush lists repeatedly.
- Shared mutable lists can cause subtle state corruption across functions.

In [ ]:
records = ["job1", "job2", "job3"]
print(records[0])
print(len(records))

## 2. Minimal Concept
Core syntax:
- Create with `[]`
- Access by index
- Slice with `start:stop`
- Mutate with item assignment and methods like `append()`

## 3. Mental Model
How Python actually behaves
A list is an ordered, mutable container of object references. The list stores references, not embedded copies of the objects themselves. Changing the list structure and mutating objects inside the list are different operations, and both matter in production code.

In [ ]:
items = [{"id": 1}, {"id": 2}]
copy_ref = items
items.append({"id": 3})
print(items)
print(copy_ref)
print(id(items), id(copy_ref))

## 4. Internal Mechanics
Lists support indexing and slicing through sequence operations, and mutation happens in place. Identity and bytecode make it visible that appending changes the same list object rather than creating a new list automatically.

In [ ]:
import dis

def add_record(buffer, value):
    buffer.append(value)
    return buffer

dis.dis(add_record)
buffer = [1, 2]
print(id(buffer))
add_record(buffer, 3)
print(id(buffer), buffer)

## 5. Edge Cases
Where it breaks:
- Negative indexing is useful but easy to misuse under empty input.
- Shallow copies do not isolate nested objects.
- Mutating a list while iterating over it can skip elements.
- `*` with nested lists can duplicate references, not structures.

In [ ]:
matrix = [[0] * 3] * 2
matrix[0][0] = 99
print(matrix)

values = [1, 2, 3, 4]
for value in values:
    if value % 2 == 0:
        values.remove(value)
print(values)

## 6. Performance Thinking
- Index access is O(1).
- Appending at the end is amortized O(1).
- Inserting or deleting near the front is O(n).
- Membership checks like `x in list` are O(n), which matters at scale.

## 7. Real Use Case
A data pipeline often collects validated records into a batch list before sending them to storage in one write.

In [ ]:
incoming = [{"id": 1}, {"id": 2}, {"id": 3}]
batch = []
for record in incoming:
    batch.append(record)
print(batch)

## 8. Anti-Pattern
What beginners do wrong:
- Use a list where set or dict lookup is the real need.
- Mutate a list in place without considering aliases.
- Assume copying the outer list isolates all nested values.

In [ ]:
original = [[1], [2]]
shallow = original[:]
shallow[0].append(99)
print(original)
print(shallow)

## 9. Interview Signals
Questions FAANG asks:
- Why is append fast but insert at the front slow?
- What is the difference between shallow copy and deep copy for lists?
- Why is `[[0] * 3] * 2` a bug pattern?
- When should a list be replaced with a set or dict?

## 10. Exercise (Non-trivial)
Design a batch accumulator for incoming API records. It should accept new records, flush once it reaches a capacity, avoid aliasing bugs between retries, and explain where list operations are efficient versus where another structure would be safer.

In [ ]:
def add_to_batch(batch, record, limit):
    batch.append(record)
    if len(batch) >= limit:
        flushed = batch[:]
        batch.clear()
        return flushed
    return None

# Task:
# 1. Explain why slicing before clear matters.
# 2. Identify the complexity of append and clear.
# 3. Describe a retry-safe version of this design.